In [ ]:
import tensorflow_datasets as tfds
import numpy as np
import keras
import wandb
from wandb.integration.keras import WandbMetricsLogger
import tensorflow as tf
from keras.layers import Normalization, Dense, Input, Conv2D, MaxPool2D, Flatten, BatchNormalization, RandomRotation, RandomFlip
from keras.losses import BinaryCrossentropy
from keras.optimizers import Adam
from keras.metrics import BinaryAccuracy, TruePositives, TrueNegatives, FalsePositives, FalseNegatives, Precision, Recall, AUC

In [ ]:
dataset, dataset_info = tfds.load("malaria", with_info = True, split = "train", as_supervised = True)

In [ ]:
def split(dataset, train_size, val_size):
    dataset_size = len(dataset)
    train_dataset = dataset.take(int(dataset_size*train_size))
    val_test_dataset = dataset.skip(int(dataset_size*train_size))
    val_dataset = val_test_dataset.take(int(dataset_size*val_size))
    test_dataset = val_test_dataset.skip(int(dataset_size*val_size))
    return train_dataset, val_dataset, test_dataset

In [ ]:
global train_ds, val_ds, test_ds
train_ds, val_ds, test_ds = split(dataset, 0.8, 0.1)

In [ ]:
wandb.login(key='2406546a8697702538ae05f97ebcb90672d1e719')

wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: shahrozhanif32 (shahrozhanif32-university-of-engineering-and-technology-). Use `wandb login --relogin` to force relogin
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [ ]:
def create_model(config):
    """Create model with given hyperparameters"""
    model = tf.keras.Sequential([
        Input(shape=(config.im_size, config.im_size, 3)),
        BatchNormalization(),
        Conv2D(filters=config.initial_filters, kernel_size=config.kernel_size, strides=1, padding='valid', activation="relu", use_bias=False),
        BatchNormalization(),
        MaxPool2D(pool_size=2, strides=2),
        Conv2D(filters=config.initial_filters * 2, kernel_size=config.kernel_size, strides=1, padding='valid', activation="relu", use_bias=False),
        BatchNormalization(),
        MaxPool2D(pool_size=2, strides=2),
        Conv2D(filters=config.initial_filters * 4, kernel_size=config.kernel_size, strides=1, padding='valid', activation="relu", use_bias=False),
        BatchNormalization(),
        MaxPool2D(pool_size=2, strides=2),
        Flatten(),
        Dense(config.dense_units, activation="relu"),
        BatchNormalization(),
        Dense(config.dense_units // 10, activation="relu"),
        BatchNormalization(),
        Dense(1, activation="sigmoid")
    ])

    return model

In [ ]:
def train():
    # Initialize wandb
    wandb.init()

    # Access hyperparameters through wandb.config
    config = wandb.config

    # Preprocessing layers
    resize_layer = tf.keras.layers.Resizing(config.im_size, config.im_size)
    rescale_layer = tf.keras.layers.Rescaling(1/255)

    def resize_rescale(image, label):
        image = resize_layer(image)
        image = rescale_layer(image)
        return image, label

    # Dataset preparation (assuming train_dataset, val_dataset, test_dataset are globally available)
    train_dataset = train_ds.map(resize_rescale)
    val_dataset = val_ds.map(resize_rescale)
    test_dataset = test_ds.map(resize_rescale)

    train_dataset = train_dataset.shuffle(buffer_size=162).batch(config.batch_size)
    val_dataset = val_dataset.shuffle(buffer_size=162).batch(config.batch_size)
    test_dataset = test_dataset.batch(1)

    # Create and compile model
    model = create_model(config)
    metrics = [
        BinaryAccuracy(),
        TruePositives(),
        TrueNegatives(),
        FalsePositives(),
        FalseNegatives(),
        Precision(),
        Recall()
    ]

    optimizer = getattr(tf.keras.optimizers, config.optimizer)
    optimizer = optimizer(learning_rate=config.learning_rate)

    model.compile(
        optimizer=optimizer,
        loss=BinaryCrossentropy(),
        metrics=metrics
    )

    # Early stopping callback
    early_stopping = tf.keras.callbacks.EarlyStopping(
        monitor='val_binary_accuracy',
        min_delta=0.01,            # Minimum change to count as an improvement
        patience=3,                # Number of epochs to wait before stopping
        restore_best_weights=True, # Restore model weights from the epoch with the best value
        baseline=0.5,             # Training will stop if the model doesn't show at least this value
        mode='max'                # We're tracking accuracy, so we want to maximize
    )

    # Train the model
    history = model.fit(
        train_dataset,
        validation_data=val_dataset,
        epochs=config.n_epochs,
        verbose=1,
        callbacks=[
            WandbMetricsLogger(),
            early_stopping,
        ]
    )

    # Log final metrics
    val_loss, val_accuracy = model.evaluate(val_dataset)
    wandb.log({
        "final_val_loss": val_loss,
        "final_val_accuracy": val_accuracy
    })

In [ ]:
# Define sweep configuration
sweep_config = {
    'method': 'bayes',
    'metric': {
        'name': 'val_binary_accuracy',
        'goal': 'maximize'
    },
    'early_terminate': {
        'type': 'hyperband',
        'min_iter': 3,
        's': 3,
    },
    'parameters': {
        'learning_rate': {
            'distribution': 'log_uniform_values',
            'min': 0.0001,
            'max': 0.01
        },
        'batch_size': {
            'values': [16, 32, 64]
        },
        'n_epochs': {
            'value': 10
        },
        'optimizer': {
            'values': ['Adam', 'RMSprop']
        },
        'im_size': {
            'values': [128, 256]
        },
        'kernel_size': {
            'values': [3, 5]
        },
        'initial_filters': {
            'values': [4, 8]
        },
        'dense_units': {
            'values': [64, 100, 128]
        }
    }
}

In [ ]:
# Initialize sweep
sweep_id = wandb.sweep(sweep_config, project='my-project')

# Run sweep
wandb.agent(sweep_id, train, count=20)  # Run 20 trials

Create sweep with ID: zbmpipvk
Sweep URL: https://wandb.ai/shahrozhanif32-university-of-engineering-and-technology-/my-project/sweeps/zbmpipvk


wandb: Agent Starting Run: 57ikhqb6 with config:
wandb: 	batch_size: 64
wandb: 	dense_units: 64
wandb: 	im_size: 256
wandb: 	initial_filters: 4
wandb: 	kernel_size: 5
wandb: 	learning_rate: 0.001739629919291018
wandb: 	n_epochs: 10
wandb: 	optimizer: Adam


Epoch 1/10
345/345 ━━━━━━━━━━━━━━━━━━━━ 79s 193ms/step - binary_accuracy: 0.7700 - false_negatives: 1154.6012 - false_positives: 958.6272 - loss: 0.4722 - precision: 0.7833 - recall: 0.7523 - true_negatives: 4584.1704 - true_positives: 4406.2197 - val_binary_accuracy: 0.6613 - val_false_negatives: 12.0000 - val_false_positives: 921.0000 - val_loss: 1.2917 - val_precision: 0.5997 - val_recall: 0.9914 - val_true_negatives: 442.0000 - val_true_positives: 1380.0000
Epoch 2/10
345/345 ━━━━━━━━━━━━━━━━━━━━ 64s 162ms/step - binary_accuracy: 0.9383 - false_negatives: 299.4971 - false_positives: 380.0347 - loss: 0.1756 - precision: 0.9333 - recall: 0.9452 - true_negatives: 5163.0146 - true_positives: 5261.0723 - val_binary_accuracy: 0.9390 - val_false_negatives: 25.0000 - val_false_positives: 143.0000 - val_loss: 0.1696 - val_precision: 0.9053 - val_recall: 0.9820 - val_true_negatives: 1220.0000 - val_true_positives: 1367.0000
Epoch 3/10


wandb: Ctrl + C detected. Stopping sweep.
